# Saavdhan: Evon on a Kaggle GPU

Runs Gnani's **Evon v3.3 30B** with llama.cpp on Kaggle's free **2x T4** and publishes it on the same ngrok address
the Saavdhan app already uses, so nothing changes on the laptop.

**Before Run All:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Add-ons → Secrets: `NGROK_AUTHTOKEN` and
`EVON_API_KEY` (the `LLM_API_KEY` value from Saavdhan's `.env`), both attached to this notebook.

Only one ngrok agent can hold the address: if the GPU box comes back with its own ngrok running, stop one of them.

In [ ]:
# 1. What we have: GPUs and free disk
import glob, json, os, shutil, subprocess, time, urllib.request
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", shell=True, capture_output=True, text=True).stdout)
free_gb = shutil.disk_usage("/tmp").free / 1e9
print(f"free disk in /tmp: {free_gb:.0f} GB")

In [ ]:
# 2. Secrets (never printed)
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
NGROK_AUTHTOKEN = _s.get_secret("NGROK_AUTHTOKEN")
EVON_API_KEY = _s.get_secret("EVON_API_KEY")
NGROK_DOMAIN = "YOUR-STATIC-DOMAIN.ngrok-free.dev"
print("secrets loaded")

In [ ]:
# 3. Build llama.cpp for the T4 (15-25 minutes, before the model is loaded so they don't compete for RAM).
# The ready-made Linux CUDA build ran on the CPU on Kaggle (0.8 tokens/s), and CMake can't find Kaggle's
# GPU driver library by itself, so we point it there.
drv = subprocess.run("ldconfig -p | grep -m1 'libcuda.so.1' | awk '{print $NF}'", shell=True,
                     capture_output=True, text=True).stdout.strip()
if not drv:
    found = glob.glob("/usr/local/nvidia/lib64/libcuda.so*") + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*") + \
            glob.glob("/usr/local/cuda/lib64/stubs/libcuda.so")
    drv = found[0] if found else ""
print("driver library:", drv or "NOT FOUND")
t = time.time()
r = subprocess.run("cd /tmp && rm -rf src && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp src && cd src && "
                   "cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF "
                   f"-DCUDA_cuda_driver_LIBRARY={drv} > /tmp/build.log 2>&1 && "
                   "cmake --build build -j4 --target llama-server >> /tmp/build.log 2>&1", shell=True)
print(f"build {'OK' if r.returncode == 0 else 'FAILED'} in {(time.time() - t) / 60:.0f} min")
print(open("/tmp/build.log").read()[-800:])
SERVER, ENV = "/tmp/src/build/bin/llama-server", dict(os.environ)

In [ ]:
# 4. Evon: Q4_K_M (24.7 GB) fits fully on the two T4s; IQ4_XS (18.4 GB) if disk is short
from huggingface_hub import hf_hub_download
name = "gnani-evon-v3.3-30B-A3B-Q4_K_M.gguf" if free_gb > 30 else "gnani-evon-v3.3-30B-A3B-IQ4_XS.gguf"
t = time.time()
MODEL = hf_hub_download("luminenio/gnani-evon-v3.3-30B-A3B-GGUF", name, local_dir="/tmp/evon")
print(MODEL, f"{os.path.getsize(MODEL) / 1e9:.1f} GB in {time.time() - t:.0f} s")

In [ ]:
# 5. Start Evon (password-protected) and wait until it has loaded
log = open("/tmp/llama.log", "w")
SRV = subprocess.Popen([SERVER, "-m", MODEL, "--alias", "evon", "--jinja", "-ngl", "99", "-c", "16384",
                        "--host", "127.0.0.1", "--port", "8080", "--api-key", EVON_API_KEY],
                       stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, env=ENV,
                       start_new_session=True)  # own process group: stopping a cell must not kill Evon
for i in range(240):
    time.sleep(5)
    if SRV.poll() is not None:
        print("Evon stopped while loading:")
        break
    try:
        if b'"ok"' in urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5).read():
            print(f"Evon loaded in {(i + 1) * 5} s")
            break
    except Exception:
        pass
print("\n".join(l for l in open("/tmp/llama.log").read().splitlines() if "offload" in l or "CUDA" in l)[:1500]
      or open("/tmp/llama.log").read()[-1500:])

In [ ]:
# 6. Speed check (expect 30+ tokens/s on the GPUs; under 2 means it is running on the CPU): one short Hindi verdict
body = {"model": "evon", "temperature": 0, "max_tokens": 120, "chat_template_kwargs": {"enable_thinking": False},
        "messages": [{"role": "user", "content": "Reply in one line: is this a scam? 'Aapka KYC expire ho gaya, OTP bataiye'"}]}
req = urllib.request.Request("http://127.0.0.1:8080/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json", "Authorization": f"Bearer {EVON_API_KEY}"})
d = json.load(urllib.request.urlopen(req, timeout=300))
print(d["choices"][0]["message"]["content"])
print(f"{d['timings']['predicted_per_second']:.1f} tokens/s")

In [ ]:
# 7. Publish on the same ngrok address the app already uses.
# pyngrok only installs the ngrok program; we run it directly, because newer ngrok renamed "domain" to "url".
subprocess.run("pip install -q pyngrok", shell=True, check=True)
from pyngrok import ngrok, conf
ngrok.install_ngrok()
ngrok.kill()
NGROK = conf.get_default().ngrok_path

def start_tunnel(flag):
    return subprocess.Popen([NGROK, "http", "127.0.0.1:8080", f"{flag}=https://{NGROK_DOMAIN}",
                             "--authtoken", NGROK_AUTHTOKEN, "--log", "stdout"],
                            stdout=open("/tmp/ngrok.log", "w"), stderr=subprocess.STDOUT,
                            stdin=subprocess.DEVNULL, start_new_session=True)

TUN = start_tunnel("--url")
live = False
for i in range(30):
    time.sleep(2)
    log = open("/tmp/ngrok.log").read()
    if "unknown flag" in log and TUN.poll() is not None:   # older ngrok: the flag is --domain
        TUN = start_tunnel("--domain")
        continue
    try:
        t = json.load(urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels", timeout=3))["tunnels"]
        if t:
            print("Evon is live at", t[0]["public_url"])
            live = True
            break
    except Exception:
        pass
    if TUN.poll() is not None:
        break
if not live:
    print("ngrok did not start:\n", open("/tmp/ngrok.log").read()[-1200:])

In [ ]:
# 8. Keep the session alive; restart the ngrok link by itself if it drops (stop this cell to shut down).
# Interrupting a cell can take the ngrok process down with it, so don't interrupt this one while in use.
while True:
    time.sleep(60)
    if SRV.poll() is not None:
        print(time.strftime("%H:%M"), "Evon STOPPED:\n", open("/tmp/llama.log").read()[-1500:])
        break
    if TUN.poll() is not None:
        print(time.strftime("%H:%M"), "link dropped, restarting ngrok", flush=True)
        TUN = start_tunnel("--url")
        time.sleep(5)
    print(time.strftime("%H:%M"), "Evon running, link up", flush=True)